# LLM Fundamentals — Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

**1. Token Estimation and Cost Calculation.** You're building a feature that summarizes customer reviews. Each review averages 800 characters, and you expect the summary to be about 60 tokens.

In [ ]:
import math

# US$ per 1M tokens (input, output) — Anthropic price table
PRICES = {
    "haiku":  (1.0, 5.0),
    "sonnet": (2.0, 10.0),
    "opus":   (5.0, 25.0),
}
CHARS_PER_TOKEN = 4  # English-prose heuristic

def estimate_monthly_cost(reviews_per_day, chars_per_review, output_tokens, model):
    """Estimate monthly cost (30 days) for processing reviews."""
    p_in, p_out = PRICES[model]
    in_tokens = math.ceil(chars_per_review / CHARS_PER_TOKEN)
    daily_in = in_tokens * reviews_per_day
    daily_out = output_tokens * reviews_per_day
    daily_cost = daily_in / 1e6 * p_in + daily_out / 1e6 * p_out
    return daily_cost * 30

# Task 2: Monthly cost for 5,000 reviews/day on each tier
print("Monthly cost for 5,000 reviews/day:")
for tier in ["haiku", "sonnet", "opus"]:
    cost = estimate_monthly_cost(5000, 800, 60, tier)
    print(f"  {tier:>8}: ${cost:,.2f}")

# Task 3: How many reviews for $1,000/month on Haiku vs Opus?
print("\nReviews processable for $1,000/month:")
for tier in ["haiku", "opus"]:
    p_in, p_out = PRICES[tier]
    cost_per_review = (800 / 4 / 1e6 * p_in + 60 / 1e6 * p_out)
    reviews = int(1000 / 30 / cost_per_review)
    print(f"  {tier:>8}: {reviews:,} reviews/day")

**2. Understanding Training Stages.** Identify which training stage is primarily responsible for each behavior.

In [ ]:
training_stages = {
    1: "pretraining",      # Python syntax learned from code pretraining
    2: "RLHF",             # Safety training via reinforcement learning
    3: "RLHF",             # Chat format trained via RLHF
    4: "pretraining",      # World knowledge from pretraining corpus
    5: "RLHF",             # Professional tone shaped by RLHF
    6: "pretraining"       # Creative writing exposure in pretraining
}

print("Training stages analysis:")
for num, stage in training_stages.items():
    print(f"  {num}: {stage}")

**3. Implementing Softmax with Temperature.** Implement a complete sampling function with temperature control.

In [ ]:
import numpy as np

def softmax_with_temperature(logits, temperature=1.0):
    """Apply softmax with temperature scaling."""
    shifted = logits - np.max(logits)  # numerical stability
    scaled = shifted / temperature
    exp = np.exp(scaled)
    return exp / exp.sum()

def sample_token(probabilities, rng):
    """Sample one token from a probability distribution."""
    return rng.choice(len(probabilities), p=probabilities)

vocabulary = ["sunny", "cloudy", "rainy", "stormy"]
logits = np.array([4.0, 2.5, 1.0, 0.2])

for temp in [0.1, 1.0, 2.5]:
    probs = softmax_with_temperature(logits, temp)
    rng = np.random.default_rng(42)
    samples = [vocabulary[sample_token(probs, rng)] for _ in range(5)]
    print(f"T={temp}: probs={[f'{p:.2f}' for p in probs]}")
    print(f"  samples: {samples}\n")

**4. Top-p (Nucleus) Sampling.** Implement top-p filtering to keep only the smallest set of tokens covering probability mass p.

In [ ]:
import numpy as np

def apply_top_p(probs, p):
    """Filter distribution to keep tokens covering cumulative probability >= p."""
    order = np.argsort(probs)[::-1]
    cum = np.cumsum(probs[order])
    cutoff = np.searchsorted(cum, p, side='right')
    mask = np.zeros_like(probs, dtype=bool)
    mask[order[:cutoff + 1]] = True
    filtered = np.where(mask, probs, 0.0)
    total = filtered.sum()
    if total > 0:
        filtered /= total
    return filtered

probs = np.array([0.5, 0.25, 0.15, 0.06, 0.04])
tokens = ["A", "B", "C", "D", "E"]

for p in [0.5, 0.75, 0.95, 1.0]:
    result = apply_top_p(probs, p)
    survivors = [tokens[i] for i, v in enumerate(result) if v > 0]
    print(f"p={p}: kept={survivors}, probs={[f'{v:.3f}' for v in result if v > 0]}")

**5. Model Selection Decision Matrix.** Recommend a model tier for three different features based on task, volume, and budget.

In [ ]:
def recommend_model(complexity, daily_volume, needs_accuracy):
    """Recommend a model tier based on task characteristics."""
    if complexity == "high" or needs_accuracy > 0.95:
        return "opus", "Requires deep reasoning and highest accuracy"
    if daily_volume > 1_000_000:
        return "haiku", "High volume, simple task — minimize cost"
    if complexity == "medium" or daily_volume > 100_000:
        return "sonnet", "Balanced quality and cost for moderate volume"
    return "sonnet", "Good default for prototype and low volume"

features = [
    ("Email Autoresponder", "low", 2_000_000, 0.85),
    ("Legal Doc Analysis", "high", 50, 0.98),
    ("Code Review Assistant", "medium", 800, 0.90),
]

print("Model Recommendations:")
for name, cx, vol, acc in features:
    model, reason = recommend_model(cx, vol, acc)
    print(f"  {name:<28} -> {model:>6}: {reason}")

**6. Entropy and Uncertainty.** Implement Shannon entropy and assess confidence levels.

In [ ]:
import numpy as np

def calculate_entropy(probs):
    """Compute Shannon entropy: 0 = certain, higher = more uncertain."""
    p = np.asarray(probs, dtype=float)
    p = p[p > 0]  # avoid log(0)
    return float(-np.sum(p * np.log(p)))

def assess_confidence(probs, threshold=1.0):
    """Return 'confident' if entropy < threshold, else 'uncertain'."""
    return "confident" if calculate_entropy(probs) < threshold else "uncertain"

distributions = {
    "Confident": [0.95, 0.03, 0.01, 0.01],
    "Uncertain": [0.4, 0.3, 0.2, 0.1],
    "Uniform":   [0.25, 0.25, 0.25, 0.25],
}

print("Entropy Analysis:")
for name, probs in distributions.items():
    ent = calculate_entropy(probs)
    label = assess_confidence(probs)
    print(f"  {name:<12} entropy={ent:.3f} -> {label}")

print("\nHigh entropy indicates hallucination risk because the model")
print("is guessing among many equally plausible continuations.")

**7. Tokenization Analysis.** Different text types tokenize differently — the /4 heuristic works for prose but breaks for code and JSON.

In [ ]:
import math

DIVISORS = {
    "prose":   4.0,
    "code":    3.0,
    "json":    2.5,
    "numbers": 1.5,
}

def estimate_tokens(text, text_type):
    """Estimate tokens based on text type."""
    divisor = DIVISORS.get(text_type, 4.0)
    return math.ceil(len(text) / divisor)

samples = {
    "prose":   "The quick brown fox jumps over the lazy dog near the river bank.",
    "code":    "def f(x):\n    return sum(v * v for v in x if v % 2 == 0)",
    "json":    '{"user_id": 8842, "active": true, "roles": ["admin", "editor"]}',
    "numbers": "4815162342951387",
}

print(f"{'type':<10} {'chars':>5} {'prose_est':>10} {'actual_est':>10}")
for label, text in samples.items():
    prose_est = math.ceil(len(text) / 4)
    actual_est = estimate_tokens(text, label)
    print(f"{label:<10} {len(text):>5} {prose_est:>10} {actual_est:>10}")

# Cost difference for 100k requests on Sonnet
price_in = 2.0 / 1e6
json_text = samples["json"]
prose_underestimate = math.ceil(len(json_text) / 4)
json_estimate = estimate_tokens(json_text, "json")
cost_diff = (json_estimate - prose_underestimate) * 100_000 * price_in
\nprint(f"\nCost difference for 100k JSON requests on Sonnet: ${cost_diff:.2f}")

**8. Attention Masking Mechanics.** Implement causal attention masking for autoregressive generation.

In [ ]:
import numpy as np

def create_causal_mask(seq_len):
    """Return True where position i should NOT attend to position j."""
    return np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)

seq_len = 4
scores = np.random.default_rng(42).normal(size=(seq_len, seq_len))
mask = create_causal_mask(seq_len)

# Apply mask: set future positions to -inf
scores[mask] = -np.inf

# Apply softmax row-wise
exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
weights = exp / exp.sum(axis=-1, keepdims=True)

tokens = ["The", "cat", "sat", "down"]
print("Attention weights (each token only attends to itself and earlier):")
for t, row in zip(tokens, weights.round(3)):
    print(f"  {t:<5} -> {row}")

**9. Open vs Closed Weights Decision Framework.** Build a decision helper for the open vs closed weights choice.

In [ ]:
def recommend_route(data_sensitivity, daily_volume, has_ml_ops, needs_fine_tuning):
    """Recommend open/closed/hybrid based on constraints."""
    reasons = []
    
    if data_sensitivity >= 8:
        reasons.append("high data sensitivity")
    if needs_fine_tuning:
        reasons.append("fine-tuning required")
    if daily_volume > 2_000_000:
        reasons.append("very high volume")
    if has_ml_ops and needs_fine_tuning:
        return "open_weights", f"Self-hosted: {', '.join(reasons)}"
    if data_sensitivity >= 8 and has_ml_ops:
        return "open_weights", f"Privacy + control: {', '.join(reasons)}"
    if data_sensitivity >= 8 and not has_ml_ops:
        return "closed_api", f"Enterprise agreement needed: {', '.join(reasons)}"
    if daily_volume > 2_000_000:
        return "hybrid", f"Bulk + escalation: {', '.join(reasons)}"
    return "closed_api", f"Fast iteration: {', '.join(reasons) or 'default'}"

scenarios = [
    ("Healthcare chatbot",  9,  10_000, False, False, 5000),
    ("Ad targeting",        3, 10_000_000, True, True, 50000),
    ("Internal docs Q&A",  7,   5_000, False, False, 2000),
]
print("Route Recommendations:")
for name, sens, vol, ops, ft, budget in scenarios:
    route, reason = recommend_route(sens, vol, ops, ft)
    print(f"  {name:<20} -> {route:<12} {reason}")

**10. Cost Optimization Strategy.** Design a cascade strategy for a ticket classifier.

In [ ]:
import math

PRICES = {"haiku": (1, 5), "sonnet": (2, 10), "opus": (5, 25)}

def cost_per_request(model, in_chars, out_tokens):
    p_in, p_out = PRICES[model]
    return math.ceil(in_chars / 4) / 1e6 * p_in + out_tokens / 1e6 * p_out

# Current: 100% Opus, 50k tickets/day, 600 chars in, 25 tokens out
daily = 50_000
chars, out_tok = 600, 25
current_cost = cost_per_request("opus", chars, out_tok) * daily * 30
print(f"Current monthly cost (100% Opus): ${current_cost:,.0f}")

# Cascade: 75% Haiku, 20% Sonnet, 5% Opus
cascade_cost = (
    0.75 * cost_per_request("haiku", chars, out_tok) +
    0.20 * cost_per_request("sonnet", chars, out_tok) +
    0.05 * cost_per_request("opus", chars, out_tok)
) * daily * 30
savings = (1 - cascade_cost / current_cost) * 100

print(f"Cascade monthly cost:      ${cascade_cost:,.0f}")
print(f"Savings:                   {savings:.1f}%")

# Function structure
def classify_with_cascade(ticket, confidence_threshold=0.8):
    """Classify with tier escalation."""
    result = call_llm("haiku", ticket)
    if result.confidence >= confidence_threshold:
        return result
    result = call_llm("sonnet", ticket)
    if result.confidence >= confidence_threshold:
        return result
    return call_llm("opus", ticket)

**11. Hallucination Detection Heuristic.** Build a simple system based on entropy and token probability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def entropy(probs):
    p = np.asarray(probs, dtype=float)
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)))

rng = np.random.default_rng(42)
n_tokens = 20
entropies, token_probs, risky = [], [], []

for _ in range(n_tokens):
    probs = rng.dirichlet(np.ones(5))
    ent = entropy(probs)
    chosen_idx = rng.choice(len(probs), p=probs)
    chosen_prob = probs[chosen_idx]
    
    entropies.append(ent)
    token_probs.append(chosen_prob)
    risky.append(ent > 2.0 or chosen_prob < 0.3)

risk_score = sum(risky) / len(risky) * 100
print(f"Hallucination risk score: {risk_score:.1f}% of tokens flagged")
print(f"High-risk positions: {[i for i, r in enumerate(risky) if r]}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4))
ax1.plot(entropies, marker='o')
ax1.axhline(2.0, color='r', linestyle='--', alpha=0.5)
ax1.set_ylabel('Entropy')
ax2.plot(token_probs, marker='s')
ax2.axhline(0.3, color='r', linestyle='--', alpha=0.5)
ax2.set_ylabel('Chosen Token Probability')
plt.tight_layout()
plt.show()

**12. Combined Temperature and Top-p.** Real inference engines apply both temperature and top-p together.

In [ ]:
import numpy as np
from collections import Counter

def sample_with_temperature_and_top_p(logits, temperature, top_p, rng):
    """Apply temperature, top-p filtering, then sample."""
    # Temperature
    z = np.asarray(logits, dtype=float) / temperature
    z -= z.max()
    probs = np.exp(z) / np.exp(z).sum()
    
    # Top-p filtering
    order = np.argsort(probs)[::-1]
    cum = np.cumsum(probs[order])
    cutoff = np.searchsorted(cum, top_p, side='right')
    mask = np.zeros(len(probs), dtype=bool)
    mask[order[:cutoff + 1]] = True
    filtered = np.where(mask, probs, 0.0)
    filtered /= filtered.sum()
    
    return rng.choice(len(probs), p=filtered)

rng = np.random.default_rng(42)
vocab = ["apple", "banana", "cherry", "date", "elderberry",
         "fig", "grape", "honeydew", "kiwi", "lemon"]
logits = np.array([5, 4, 3, 2, 1, 0, -1, -2, -3, -4], dtype=float)

for T in [0.5, 1.0, 2.0]:
    for p in [0.5, 0.9, 1.0]:
        samples = [vocab[sample_with_temperature_and_top_p(logits, T, p, rng)] for _ in range(100)]
        freq = Counter(samples)
        unique = len(freq)
        print(f"T={T}, p={p}: unique={unique}, top={freq.most_common(3)}")
    print()